# Project 3: Time Series Modeling of Air Quality in Nairobi

- **Concepts**:
    - Time-Series Analysis, NoSQL (MongoDB), Lagged Regression, AR & ARMA Models.
- **Description**: 
    - This project deals with time-indexed data. Using air quality measurements from Nairobi stored in MongoDB, we query, prepare, and forecast time series data. The project builds from simple lagged regression up to Autoregressive (AR) and ARMA models

In [1]:
# import liberaries
import warnings
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px 
import pytz
from statsmodels.graphics.tsaplots import plot_acf , plot_pacf
from pymongo.mongo_client import MongoClient
from sklearn.metrics import mean_absolute_error
from statsmodels.tsa.ar_model import AutoReg
from statsmodels.tsa.seasonal import seasonal_decompose


warnings.simplefilter(action="ignore", category=FutureWarning)



## Prepare Data 

- Data Wrangling with MongoDB 

### Connect to the local Docker container 

In [6]:
from pymongo import MongoClient
import pandas as pd

# Connect to the local Docker container
client = MongoClient("mongodb://localhost:27017/")

# Select the database and collection you just created
db = client["air-quality"]
nairobi_collection = db["nairobi"]

# Fetch the first 5 records to verify
cursor = nairobi_collection.find().limit(5)
df = pd.DataFrame(list(cursor))

print(df.head())

                        _id  sensor_id sensor_type  location    lat     lon  \
0  6ae260c882c8f6828eae4412       4927       DHT22      3981 -1.269  36.819   
1  6ae260c882c8f6828eae4413       4927       DHT22      3981 -1.269  36.819   
2  6ae260c882c8f6828eae4414       4930     pms5003      3981 -1.269  36.819   
3  6ae260c882c8f6828eae4415       4930     pms5003      3981 -1.269  36.819   
4  6ae260c882c8f6828eae4416       4930     pms5003      3981 -1.269  36.819   

                          timestamp   value_type  value  
0  2025-08-01T00:03:03.447032+00:00     humidity   72.9  
1  2025-08-01T00:03:03.447032+00:00  temperature   17.9  
2  2025-08-01T00:04:26.016901+00:00           P2   21.0  
3  2025-08-01T00:04:26.016901+00:00           P1   23.0  
4  2025-08-01T00:04:26.016901+00:00           P0   16.0  


In [16]:
df.info()
df.head()

<class 'pandas.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   _id          5 non-null      object 
 1   sensor_id    5 non-null      int64  
 2   sensor_type  5 non-null      str    
 3   location     5 non-null      int64  
 4   lat          5 non-null      float64
 5   lon          5 non-null      float64
 6   timestamp    5 non-null      str    
 7   value_type   5 non-null      str    
 8   value        5 non-null      float64
dtypes: float64(3), int64(2), object(1), str(3)
memory usage: 492.0+ bytes


,_id,sensor_id,sensor_type,location,lat,lon,timestamp,value_type,value
0,6ae260c882c8f6828eae4412,4927,DHT22,3981,-1.269,36.819,2025-08-01T00:03:03.447032+00:00,humidity,72.9
1,6ae260c882c8f6828eae4413,4927,DHT22,3981,-1.269,36.819,2025-08-01T00:03:03.447032+00:00,temperature,17.9
2,6ae260c882c8f6828eae4414,4930,pms5003,3981,-1.269,36.819,2025-08-01T00:04:26.016901+00:00,P2,21.0
3,6ae260c882c8f6828eae4415,4930,pms5003,3981,-1.269,36.819,2025-08-01T00:04:26.016901+00:00,P1,23.0
4,6ae260c882c8f6828eae4416,4930,pms5003,3981,-1.269,36.819,2025-08-01T00:04:26.016901+00:00,P0,16.0


In [8]:
nairobi_collection.find_one()

{'_id': ObjectId('6ae260c882c8f6828eae4412'),
 'sensor_id': 4927,
 'sensor_type': 'DHT22',
 'location': 3981,
 'lat': -1.269,
 'lon': 36.819,
 'timestamp': '2025-08-01T00:03:03.447032+00:00',
 'value_type': 'humidity',
 'value': 72.9}

## No. of unique locations

- Using the 2025 data (Aug-Dec), lets find the number of unique locations from which the sensor data is obtained.

In [17]:
import pandas as pd

# 1. Convert the timestamp column to actual datetime objects
df['timestamp'] = pd.to_datetime(df['timestamp'])

# 2. Filter for the year 2025 (matching your data)
df_2025 = df[df['timestamp'].dt.year == 2025]

# 3. Get the unique locations as a list
sites = df_2025['location'].unique().tolist()

print(sites)

[3981]
